In [93]:
import warnings
warnings.filterwarnings('ignore')

In [94]:
import pandas as pd
import numpy as np
import pyreadr
import joblib

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

from dash import Dash, html, dcc, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

In [95]:
pd.set_option('display.max_columns', 100)

In [96]:
dem_uncont = pd.read_csv('transformed/dem_uncontested_seats.csv')
rep_uncont = pd.read_csv('transformed/rep_uncontested_seats.csv')

In [97]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont.shape[0])
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 218 else 'Republicans')
seat_sims

,seats,winner
0,260,Democrats
1,217,Republicans
2,245,Democrats
3,238,Democrats
4,225,Democrats
...,...,...
19995,294,Democrats
19996,213,Republicans
19997,207,Republicans
19998,269,Democrats


In [98]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 218 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 218 else (435-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,161,1,0.005,Republicans
1,164,2,0.010,Republicans
2,166,1,0.005,Republicans
3,169,1,0.005,Republicans
4,172,2,0.010,Republicans


In [99]:
np.unique(seat_sims['seats']).shape[0]

186

In [100]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'}, template='plotly_white')
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=217.5, line_width=1, line_color='black', annotation_text='218 seats required for majority', 
                    annotation_position='top right')
sims_hist

In [101]:
output_ts = pd.read_csv('model_output/output_over_time.csv')
output_ts.head()

,date,y,geo,type
0,2026-08-30,229.365800,US House,seats
1,2026-08-30,66.120000,US House,chance
2,2026-08-30,22.784344,US House,seats_sd
3,2026-08-30,44.972806,AK-AL,y_pred
4,2026-08-30,3.555839,AK-AL,y_pred_sd


In [102]:
chance_over_time = output_ts[(output_ts['geo'] == 'US House') &
                             (output_ts['type'] == 'chance')]
chance_over_time['rep_chance'] = chance_over_time['y'].map(lambda x: 100 - x)
chance_over_time = chance_over_time.rename({'y': 'Democrats', 'rep_chance': 'Republicans'}, axis=1)
chance_time_ser = px.line(chance_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
chance_time_ser.update_traces(hovertemplate="%{y:.1f}%")
chance_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-08-30'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Win Probability (%)',
    title=dict(text="House Win Probability Over Time"),
    hovermode="x",
    showlegend=False
)
chance_time_ser.add_vline(x=pd.to_datetime('2026-10-22').timestamp() * 1000, line_width=1, line_color='black',
                          annotation_text='Poll average methodology update', annotation_position='top right')
chance_time_ser

In [103]:
seats_over_time = output_ts[(output_ts['geo'] == 'US House') &
                             (output_ts['type'] == 'seats')]
seats_over_time['rep_seats'] = seats_over_time['y'].map(lambda x: 435 - x)
seats_over_time = seats_over_time.rename({'y': 'Democrats', 'rep_seats': 'Republicans'}, axis=1)
seats_time_ser = px.line(seats_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                        color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
seats_time_ser.update_traces(hovertemplate="%{y:.1f}")
seats_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-08-30'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 435]),
    xaxis_title='Date',
    yaxis_title='Average Seats Over All Simulations',
    title=dict(text="Projected Seats Over Time"),
    hovermode="x",
    showlegend=False
)
seats_time_ser.add_vline(x=pd.to_datetime('2026-10-22').timestamp() * 1000, line_width=1, line_color='black',
                          annotation_text='Poll average methodology update', annotation_position='top right')
seats_time_ser

In [104]:
joblib.dump(sims_hist, 'display_data/sims_histogram.pkl')
joblib.dump(chance_time_ser, 'display_data/chance_time_ser.pkl')
joblib.dump(seats_time_ser, 'display_data/seats_time_ser.pkl')

['display_data/seats_time_ser.pkl']

In [105]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [106]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,19950,19951,19952,19953,19954,19955,19956,19957,19958,19959,19960,19961,19962,19963,19964,19965,19966,19967,19968,19969,19970,19971,19972,19973,19974,19975,19976,19977,19978,19979,19980,19981,19982,19983,19984,19985,19986,19987,19988,19989,19990,19991,19992,19993,19994,19995,19996,19997,19998,19999
AK-AL,0.192463,-4.884659,-2.874331,-7.753798,-4.556784,-0.194993,-3.027942,-4.595134,-2.738673,-5.619012,0.117064,1.762965,1.047380,-9.196307,6.004646,0.180693,-6.104064,-3.232722,-0.362348,-0.272323,-6.896204,-8.059678,0.598491,-6.802105,-6.696007,-1.751550,-3.104874,-5.490257,-2.090433,-10.142161,-1.104905,-5.438027,-6.272653,1.341482,-3.075422,-4.123613,-5.736508,-5.660937,-2.985714,0.179625,-10.563609,-1.884669,-7.644141,-4.247419,-1.770701,-6.744495,-1.683725,-5.392852,-4.007118,-5.271551,...,-4.561771,-5.361133,-2.575328,-2.757332,-6.320809,-1.872038,-3.607042,-2.699265,-1.629108,-0.568720,-2.720622,-0.772622,-9.973793,2.508143,-3.362686,-6.638668,-0.414092,-1.783176,-1.796380,-4.325382,-3.782625,-4.879256,-2.552948,-6.795986,-2.893444,-3.610118,-0.521693,-5.078296,-0.593248,-7.558259,-1.865598,-4.034722,-2.625809,0.482971,-0.666089,-0.738538,-1.162538,-4.362321,-1.759395,-1.830242,-7.630737,2.604788,0.856571,-4.942152,-3.654787,2.576226,-10.507264,-9.260001,-3.778071,-4.515341
AL-01,-8.873468,-13.733502,-11.834213,-14.395130,-11.492856,-7.563444,-12.413872,-12.554059,-12.219819,-13.975921,-11.573717,-12.236997,-9.292604,-14.718206,-4.047791,-9.572447,-12.869364,-12.139199,-16.215378,-8.805956,-16.745150,-16.211003,-6.434029,-19.807186,-15.197241,-8.858457,-13.600125,-16.190174,-11.985207,-18.089854,-11.418310,-15.485505,-19.478920,-9.356664,-13.079638,-13.009489,-14.015999,-18.441030,-14.810415,-4.613354,-20.285306,-9.376932,-14.884329,-12.356841,-13.218630,-18.250561,-10.862437,-14.604451,-11.492326,-12.230209,...,-12.442458,-11.561503,-13.546885,-9.540548,-12.861146,-12.600879,-12.151406,-14.606871,-14.036575,-9.172371,-9.734182,-10.222430,-17.726877,-8.882239,-10.414386,-20.419920,-4.287514,-13.435579,-16.779109,-16.419015,-10.258997,-13.021691,-14.006928,-10.270618,-12.729233,-14.741722,-10.131659,-15.035378,-9.988376,-15.059809,-8.901825,-14.052690,-14.454897,-8.815914,-7.478258,-9.134146,-11.066798,-14.073453,-15.516757,-12.916671,-14.507335,-10.795061,-12.482252,-11.284012,-17.095974,-6.279654,-17.237702,-18.952224,-8.177548,-14.104747
AL-02,-1.233218,-0.993768,0.710910,0.898309,-5.099756,-4.474356,-2.763846,-1.679431,3.176681,-3.084563,-0.389462,0.331947,0.396882,0.259413,-6.094139,-5.043774,4.106132,-5.299448,-1.654392,4.601360,-2.097854,-2.801809,1.803873,-4.707588,0.426722,-1.492527,5.443237,-3.406027,2.714022,-3.687310,0.011785,0.596125,-7.251790,1.623753,-0.091656,-0.829491,-2.639798,-5.572639,-4.957529,8.463475,-8.829605,1.218863,-2.724160,-2.672865,-3.336625,0.888149,-0.394317,-2.847465,-2.110013,5.633796,...,-0.186446,-0.205182,-1.570498,-0.771725,-0.723942,0.816095,3.764918,-9.738323,-8.089137,1.606809,-0.116863,-3.198865,1.872809,-3.166028,-2.171505,-2.453324,0.269988,0.664094,0.627193,-2.774742,1.824069,-0.721651,0.768322,-3.278949,-10.013149,-1.572017,-1.361767,-1.204471,-0.730435,0.566900,-4.813290,-2.434223,-0.829093,-2.124936,0.704697,3.691198,1.996729,-2.700328,1.095610,1.064780,-1.952649,4.191802,10.586154,-7.276233,3.929553,3.991356,-2.738960,-6.076634,2.554308,-4.859312
AL-03,-16.416867,-23.383719,-18.796801,-22.677602,-21.831935,-15.441209,-20.174122,-15.007929,-21.054637,-21.150230,-15.752909,-16.804368,-14.896596,-21.256156,-11.995507,-14.647541,-22.907791,-21.528005,-20.244141,-15.600680,-23.504633,-23.990383,-15.530028,-21.868975,-22.691987,-17.793501,-20.301704,-23.881623,-20.600088,-25.417217,-17.181484,-17.609429,-22.668032,-14.644372,-20.578776,-14.013326,-22.546777,-20.606617,-20.995890,-16.104511,-29.311197,-20.116144,-20.807964,-19.037726,-19.93

In [107]:
sim_corr = post_untransp.corr()

In [108]:
sim_corr

,AK-AL,AL-01,AL-02,AL-03,AL-04,AL-05,AL-06,AL-07,AR-01,AR-02,AR-03,AR-04,AZ-01,AZ-02,AZ-03,AZ-04,AZ-05,AZ-06,AZ-07,AZ-08,AZ-09,CA-01,CA-02,CA-03,CA-05,CA-06,CA-08,CA-09,CA-10,CA-13,CA-15,CA-16,CA-17,CA-18,CA-19,CA-20,CA-21,CA-22,CA-23,CA-24,CA-25,CA-26,CA-27,CA-28,CA-30,CA-31,CA-32,CA-33,CA-35,CA-36,...,TX-25,TX-26,TX-27,TX-28,TX-29,TX-30,TX-31,TX-32,TX-33,TX-34,TX-35,TX-36,TX-37,TX-38,UT-01,UT-02,UT-03,UT-04,VA-01,VA-02,VA-03,VA-04,VA-05,VA-06,VA-07,VA-08,VA-09,VA-10,VA-11,VT-AL,WA-01,WA-02,WA-03,WA-04,WA-05,WA-06,WA-07,WA-08,WA-09,WA-10,WI-01,WI-03,WI-04,WI-05,WI-06,WI-07,WI-08,WV-01,WV-02,WY-AL
AK-AL,1.000000,0.723857,0.484782,0.742377,0.732984,0.718941,0.741338,0.498632,0.737346,0.746962,0.742645,0.739186,0.486766,0.486737,0.485177,0.505242,0.700626,0.720787,0.729790,0.725813,0.739926,0.701189,0.505361,0.516821,0.747412,0.736821,0.568996,0.507284,0.513549,0.514907,0.482598,0.698896,0.571773,0.506810,0.502723,0.701475,0.509555,0.747555,0.733692,0.510467,0.512637,0.704250,0.486788,0.510483,0.486487,0.525228,0.549617,0.511184,0.566231,0.511709,...,0.737349,0.718391,0.547394,0.511027,0.514959,0.701720,0.749486,0.700402,0.528813,0.503723,0.701586,0.535220,0.500848,0.487501,0.500201,0.733980,0.723097,0.724257,0.742795,0.552205,0.510074,0.482782,0.721886,0.739914,0.486064,0.511807,0.739788,0.482635,0.697825,0.531469,0.512024,0.513408,0.489305,0.702434,0.543516,0.487290,0.504917,0.512145,0.576341,0.497647,0.739455,0.544982,0.575416,0.739185,0.746619,0.698028,0.720783,0.736977,0.720615,0.698455
AL-01,0.723857,1.000000,0.493968,0.750393,0.742459,0.722419,0.745501,0.505134,0.745056,0.749954,0.744898,0.747998,0.493833,0.489348,0.488903,0.512341,0.708700,0.729633,0.733209,0.729249,0.745419,0.706373,0.510403,0.516661,0.749003,0.735959,0.565619,0.512929,0.519118,0.519753,0.493587,0.703637,0.569145,0.509323,0.509620,0.709782,0.513264,0.748998,0.734035,0.513594,0.518466,0.707275,0.488821,0.516025,0.487067,0.530081,0.547651,0.514898,0.569295,0.513530,...,0.742319,0.721832,0.550603,0.516081,0.518688,0.702257,0.751172,0.708166,0.539746,0.509245,0.709554,0.537250,0.499636,0.497062,0.500511,0.740132,0.725540,0.730524,0.747908,0.548164,0.509024,0.488550,0.725622,0.738346,0.492470,0.513818,0.742789,0.492762,0.705840,0.534689,0.514895,0.518194,0.488937,0.709729,0.538080,0.490688,0.508409,0.515172,0.580314,0.503713,0.746768,0.544480,0.572989,0.741916,0.751473,0.711770,0.719539,0.745913,0.726031,0.708871
AL-02,0.484782,0.493968,1.000000,0.506255,0.498952,0.487442,0.509750,0.703724,0.495677,0.505227,0.496401,0.502032,0.682897,0.676577,0.682920,0.700289,0.641315,0.487575,0.494111,0.490898,0.505757,0.638411,0.705268,0.713379,0.509900,0.501725,0.583081,0.705426,0.705024,0.526038,0.672341,0.644405,0.584264,0.703654,0.704402,0.645276,0.706924,0.508988,0.502035,0.711170,0.714742,0.646442,0.679639,0.705064,0.677858,0.551846,0.568042,0.710008,0.584344,0.712386,...,0.503079,0.492782,0.559754,0.706678,0.701643,0.642536,0.511026,0.639606,0.553092,0.701195,0.647089,0.550002,0.689859,0.683772,0.688056,0.501639,0.485196,0.495336,0.500487,0.560376,0.702319,0.676171,0.489992,0.503339,0.674478,0.712668,0.503189,0.679665,0.645046,0.542140,0.707728,0.706790,0.679158,0.645630,0.559852,0.670331,0.699551,0.705723,0.590035,0.694072,0.501160,0.557835,0.589377,0.501136,0.503583,0.645863,0.482251,0.505878,0.491495,0.643219
AL-03,0.742377,0.750393,0.506255,1.000000,0.761416,0.748196,0.773691,0.521281,0.765223,0.769842,0.763570,0.768960,0.505698,0.503584,0.500787,0.524236,0.724305,0.744503,0.746158,0.748767,0.765780,0.725081,0.525173,0.531218,0.768761,0.758230,0.582120,0.525438,0.533740,0.531169,0.503760,0.722524,0.584417,0.522176,0.524044,0.729626,0.525283,0.765319,0.753894,0.527695,0.531152,0.723741,0.501557,0.527609,0.502917,0.545660,0.566694,0.527423,0.582042,0.531886,...,0.761878,0.740927,0.563301,0.533357,0.530761,0.721098,0.773169,0.727204,0.547774,0.522610,0.727723,0.551166,0.515224,0.509696,0.515736,0.755026,0.740885,0.742557,0.768870,0.560445,0.521849,0.498025,0.74426

In [109]:
post.shape

(418, 20000)

In [110]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 218 - dem_uncont.shape[0]: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (218 - dem_uncont.shape[0])
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (435 - seats_won_dem) - (218 - rep_uncont.shape[0])
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [111]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|███████████████████████████████████████████████████████████████████████████| 20000/20000 [00:31<00:00, 631.02it/s]


array(['CA-22', 'CO-05', 'FL-07', ..., 'OH-15', 'OH-07', 'MI-07'],
      shape=(20000,), dtype='<U32')

In [112]:
data = pd.read_csv('../../model_output/house_predictions.csv')
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.000000,0.000000,0.000000,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.000000,0.000000,0.000000,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",1114009.06,1390904.93,2504913.99,44.472947,55.527053,Alabama,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,56.333429,42.767786,0.987401,-8.650667,1,0,46.035454,0.655457,47.961969,0.655457,East South Central,2026,0.0,0.0,0.0,1,102,5,1977.842975,0.655457,0.809603,1.926516,-4.392748,-5.527053,1,-11.054107,48.920002,3.491543,37.850,3,42.035894,55.744866
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",62981.54,1590746.93,1653728.47,3.808457,96.191543,Alabama,AL-03,74.925713,2.047228,0.853803,0.164703,19.752354,15.301623,-22.977818,-22.768849,-22.821092,48.894607,26.481097,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,103,5,14.504346,0.000000,0.000000,0.000000,-36.991516,-46.191543,-1,-92.383086,30.344784,3.512948,0.000,4,23.415609,37.197121
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",22154.32,625402.87,647557.19,3.421214,96.578786,Alabama,AL-04,86.997080,3.646458,0.489492,0.174054,6.737716,13.622184,-33.634273,-32.744762,-32.967139,33.245792,16.505184,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,104,3,11.704708,0.000000,0.000000,0.000000,-57.283611,-46.578786,-1,-93.157571,20.532088,3.560835,0.000,5,13.533738,27.451142


In [113]:
data['tipping_point_prob'] = data['cd'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.000000,0.000000,0.000000,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.000000,0.000000,0.000000,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",1114009.06,1390904.93,2504913.99,44.472947,55.527053,Alabama,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,56.333429,42.767786,0.987401,-8.650667,1,0,46.035454,0.655457,47.961969,0.655457,East South Central,2026,0.0,0.0,0.0,1,102,5,1977.842975,0.655457,0.809603,1.926516,-4.392748,-5.527053,1,-11.054107,48.920002,3.491543,37.850,3,42.035894,55.744866,0.840
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",62981.54,1590746.93,1653728.47,3.808457,96.191543,Alabama,AL-03,74.925713,2.047228,0.853803,0.164703,19.752354,15.301623,-22.977818,-22.768849,-22.821092,48.894607,26.481097,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,103,5,14.504346,0.000000,0.000000,0.000000,-36.991516,-46.191543,-1,-92.383086,30.344784,3.512948,0.000,4,23.415609,37.197121,0.000
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",22154.32,625402.87,647557.19,3.421214,96.578786,Alabama,AL-04,86.997080,3.646458,0.489492,0.174054,6.737716,13.622184,-33.634273,-32.744762,-32.967139,33.245792,16.505184,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,104,3,11.704708,0.000000,0.000000,0.000000,-57.283611,-46.578786,-1,-93.157571,20.532088,3.560835,0.000,5,13.533738,27.451142,0.000


In [114]:
data.sort_values('tipping_point_prob', ascending=False).head(10)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
193,193,MI-10,Christina Hines,Mike Bouchard,False,False,MI,10,"HINES, CHRISTINA","BOUCHARD, MICHAEL",3380348.82,1841619.43,5221968.25,64.733232,35.266768,Michigan,MI-10,76.551128,2.316124,5.150984,0.123322,13.269372,29.340000,-2.765891,-2.592362,-2.635744,99.999226,46.657584,0.987401,-8.650667,0,0,45.000000,0.046386,45.000000,0.046386,East North Central,2026,0.0,0.0,0.0,26,2610,2,4190.391320,0.046386,0.215375,-1.460326e-07,3.379180,14.733232,0,29.466464,51.952716,3.653897,70.460,194,44.779916,59.053594,2.935
98,98,FL-22,Pia Dandiya,Casey Askar,False,False,FL,22,"DANDIYA, PIA","ASKAR, CASEY",3601978.98,383626.04,3985605.02,90.374710,9.625290,Florida,FL-22,53.931205,26.984941,3.703933,0.199203,13.186062,27.190634,-0.817400,-4.551960,-3.618320,91.180168,44.697986,0.987401,-8.650667,0,0,46.286766,0.135652,44.430149,0.135652,South Atlantic,2026,0.0,0.0,0.0,12,1222,6,8167.588214,0.135652,0.368310,-1.856617e+00,1.414027,40.374710,0,80.749420,52.154630,3.657252,72.100,99,44.882341,59.289955,2.770
206,206,MO-02,Fred Wellman,Ann Wagner,False,True,MO,2,"WELLMAN, FREDERICK PAUL","WAGNER, ANN L.",2040179.50,2354885.63,4395065.13,46.419779,53.580221,Missouri,MO-02,89.786282,2.324956,3.134970,0.061673,3.001461,48.680000,-6.210595,-3.268235,-4.003825,87.342792,45.981711,0.987401,-8.650667,0,1,49.102647,2.435184,47.590277,2.435184,West North Central,2026,0.0,0.0,0.0,29,2902,2,2154.795840,2.435184,1.560508,-1.512370e+00,0.643017,-3.580221,-1,-7.160443,51.653932,3.477376,68.165,207,44.899324,58.492666,2.575
190,190,MI-07,William Lawrence,Tom Barrett,False,True,MI,7,"LAWRENCE, WILLIAM","BARRETT, THOMAS MORE",3804135.29,4565373.62,8369508.91,45.452312,54.547688,Michigan,MI-07,85.239895,4.803904,1.944855,0.170025,5.279273,36.270000,-2.003691,0.098125,-0.427329,68.756305,49.348071,0.987401,-8.650667,0,1,52.975701,0.684585,45.011848,0.684585,East North Central,2026,0.0,0.0,0.0,26,2607,2,2065.912652,0.684585,0.827396,-7.963853e+00,7.796010,-4.547688,-1,-9.095376,52.945309,3.559793,79.755,191,45.966299,60.002668,2.570
229,229,NC-11,Jamie Ager,Jennifer Balkcom,False,False,NC,11,"AGER, JAMIE","BALKCOM, JENNIFER",5040487.66,106191.99,5146679.65,97.936689,2.063311,North Carolina,NC-11,88.740422,3.902084,0.797632,1.120840,3.305854,24.807640,-7.843347,-4.057125,-5.003681,49.294326,45.192821,0.987401,-8.650667,0,0,56.000000,0.628314,43.000000,0.628314,South Atlantic,2026,0.0,0.0,0.0,37,3711,3,9591.595118,0.628314,0.792663,-1.300000e+01,-1.356694,47.936689,0,95.873379,52.986118,3.667504,78.975,230,45.755774,60.175542,2.490
314,314,PA-07,Bob Brooks,Ryan Mackenzie,False,True,PA,7,"BROOKS, BOB","MACKENZIE, RYAN EDWARD",4181508.28,1902194.06,6083702.34,68.732953,31.267047,Pennsylvania,PA-07,75.458946,15.481046,2.435986,0.030199,4.952274,31.050000,-1.956264,-0.869379,-1.141100,81.633883,48.380567,0.987401,-8.650667,0,1,48.746393,1.394027,43.013016,1.394027,Mid-Atlantic,2026,0.0,0.0,0.0,42,4207,2,4724.218851,1.394027,1.180689,-5.733376e+00,6.368467,18.732953,-1,37.465906,52.508263,3.571214,75.820,315,45.649667,59.555551,2.485
289,289,OH-07,Brian Poindexter,Max Miller,False,True,OH,7,"POINDEXTER, BRIAN","MILLER, MAX",3308787.44,1019666.18,4328453.62,76.442714,23.557286,Ohio,OH-07,89.432935,3.348946,2.187434,0.068208,3.218146,26.556779,-7.118675,-4.823517,-5.397307,83.893826,44.426429,0.987401,-8

In [115]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand, dem_inc_any, rep_inc_any):
    indie_d = ['Bill Hill']
    indie_r = ['Kevin Kiley']
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (I)'
        dem_color = '#792ba6'
    else:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (D)'
        dem_color = '#366bbf'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (I)'
        rep_color = '#792ba6'
    else:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (R)'
        rep_color = '#e63929'

    return f'<b style="color:{dem_color};">' + dem_lab + f'</b> vs <b style="color:{rep_color};">' + rep_lab + '</b>'

In [116]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                          x['dem_inc_any'], x['rep_inc_any']), axis=1)
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.000000,0.000000,0.000000,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065,Likely R,"<b style=""color:#792ba6;"">Bill Hill (I)</b> vs..."
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.000000,0.000000,0.000000,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000,Safe R,"<b style=""color:#366bbf;"">Clyde Jones Jr. (D)<..."
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",1114009.06,1390904.93,2504913.99,44.472947,55.527053,Alabama,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,56.333429,42.767786,0.987401,-8.650667,1,0,46.035454,0.655457,47.961969,0.655457,East South Central,2026,0.0,0.0,0.0,1,102,5,1977.842975,0.655457,0.809603,1.926516,-4.392748,-5.527053,1,-11.054107,48.920002,3.491543,37.850,3,42.035894,55.744866,0.840,Tilt R,"<b style=""color:#366bbf;"">Shomari Figures* (D)..."
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",62981.54,1590746.93,1653728.47,3.808457,96.191543,Alabama,AL-03,74.925713,2.047228,0.853803,0.164703,19.752354,15.301623,-22.977818,-22.768849,-22.821092,48.894607,26.481097,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,103,5,14.504346,0.000000,0.000000,0.000000,-36.991516,-46.191543,-1,-92.383086,30.344784,3.512948,0.000,4,23.415609,37.197121,0.000,Safe R,"<b style=""color:#366bbf;"">Lee McInnis (D)</b> ..."
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",22154.32,625402.87,647557.19,3.421214,96.578786,Alabama,AL-04,86.997080,3.646458,0.489492,0.174054,6.737716,13.622184,-33.634273,-32.744762,-32.967139,33.245792,16.505184,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,104,3,11.704708,0.000000,0.000000,0.000000,-57.283611,-46.578786,-1,-93.157571,20.532088,3.560835,0.000,5,13.533738,27.451142,0.000,Safe R,"<b style=""color:#366bbf;"">Amanda Pusczek (D)</..."


In [117]:
data['projected_winner'] = data['chance'].map(lambda x: '(D)' if x > 50 else '(R)')
data.head(3)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.000000,0.000000,0.000000,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065,Likely R,"<b style=""color:#792ba6;"">Bill Hill (I)</b> vs...",(R)
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.000000,0.000000,0.000000,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000,Safe R,"<b style=""color:#366bbf;"">Clyde Jones Jr. (D)<...",(R)
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",1114009.06,1390904.93,2504913.99,44.472947,55.527053,Alabama,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,56.333429,42.767786,0.987401,-8.650667,1,0,46.035454,0.655457,47.961969,0.655457,East South Central,2026,0.0,0.0,0.0,1,102,5,1977.842975,0.655457,0.809603,1.926516,-4.392748,-5.527053,1,-11.054107,48.920002,3.491543,37.850,3,42.035894,55.744866,0.840,Tilt R,"<b style=""color:#366bbf;"">Shomari Figures* (D)...",(R)


In [118]:
pvi_24 = pd.read_csv('../../transformed/pvi/past_pres_results_by24dist.csv')
data = pd.merge(left=data, right=pvi_24[['district', 'party']], left_on='cd', right_on='district')
data = data.rename({'party': 'curr_party'}, axis=1)
data['hold'] = data['projected_winner'] ==  data['curr_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district_x,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,district_y,curr_party,hold,flip,flip_indic
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.0,0.0,0.0,0.0,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.0,0.0,0.0,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065,Likely R,"<b style=""color:#792ba6;"">Bill Hill (I)</b> vs...",(R),AK-AL,(R),True,False,
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.0,0.0,0.0,0.0,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.0,0.0,0.0,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000,Safe R,"<b style=""color:#366bbf;"">Clyde Jones Jr. (D)<...",(R),AL-01,(R),True,False,


In [119]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district_x,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,district_y,curr_party,hold,flip,flip_indic,projected_2p_margin
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.000000,0.000000,0.000000,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065,Likely R,"<b style=""color:#792ba6;"">Bill Hill (I)</b> vs...",(R),AK-AL,(R),True,False,,R+7.2
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.000000,0.000000,0.000000,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000,Safe R,"<b style=""color:#366bbf;"">Clyde Jones Jr. (D)<...",(R),AL-01,(R),True,False,,R+25.3
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",1114009.06,1390904.93,2504913.99,44.472947,55.527053,Alabama,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,56.333429,42.767786,0.987401,-8.650667,1,0,46.035454,0.655457,47.961969,0.655457,East South Central,2026,0.0,0.0,0.0,1,102,5,1977.842975,0.655457,0.809603,1.926516,-4.392748,-5.527053,1,-11.054107,48.920002,3.491543,37.850,3,42.035894,55.744866,0.840,Tilt R,"<b style=""color:#366bbf;"">Shomari Figures* (D)...",(R),AL-02,(D),False,True,Flip,R+2.2


In [120]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district_x,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,district_y,curr_party,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.000000,0.000000,0.000000,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065,Likely R,"<b style=""color:#792ba6;"">Bill Hill (I)</b> vs...",(R),AK-AL,(R),True,False,,R+7.2,84.225,15.8,84.2,15.8%,84.2%
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.000000,0.000000,0.000000,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000,Safe R,"<b style=""color:#366bbf;"">Clyde Jones Jr. (D)<...",(R),AL-01,(R),True,False,,R+25.3,99.980,0.0,100.0,<1%,>99%
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",1114009.06,1390904.93,2504913.99,44.472947,55.527053,Alabama,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,56.333429,42.767786,0.987401,-8.650667,1,0,46.035454,0.655457,47.961969,0.655457,East South Central,2026,0.0,0.0,0.0,1,102,5,1977.842975,0.655457,0.809603,1.926516,-4.392748,-5.527053,1,-11.054107,48.920002,3.491543,37.850,3,42.035894,55.744866,0.840,Tilt R,"<b style=""color:#366bbf;"">Shomari Figures* (D)...",(R),AL-02,(D),False,True,Flip,R+2.2,62.150,37.8,62.2,37.8%,62.2%


In [121]:
data['swing_24_to_26'] = data['y_pred'].astype(float).map(lambda x: x - (100 - x)) - data['dem_2p_24'].astype(float).map(lambda x: x - (100 - x))
data.head(3)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district_x,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,district_y,curr_party,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.000000,0.000000,0.000000,0.000000,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.000000,0.000000,0.000000,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065,Likely R,"<b style=""color:#792ba6;"">Bill Hill (I)</b> vs...",(R),AK-AL,(R),True,False,,R+7.2,84.225,15.8,84.2,15.8%,84.2%,6.487092
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.000000,0.000000,0.000000,0.000000,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.000000,0.000000,0.000000,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000,Safe R,"<b style=""color:#366bbf;"">Clyde Jones Jr. (D)<...",(R),AL-01,(R),True,False,,R+25.3,99.980,0.0,100.0,<1%,>99%,10.997150
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",1114009.06,1390904.93,2504913.99,44.472947,55.527053,Alabama,AL-02,53.493739,2.691740,1.032138,0.183181,40.524512,16.170153,-6.640350,-6.482160,-6.521708,56.333429,42.767786,0.987401,-8.650667,1,0,46.035454,0.655457,47.961969,0.655457,East South Central,2026,0.0,0.0,0.0,1,102,5,1977.842975,0.655457,0.809603,1.926516,-4.392748,-5.527053,1,-11.054107,48.920002,3.491543,37.850,3,42.035894,55.744866,0.840,Tilt R,"<b style=""color:#366bbf;"">Shomari Figures* (D)...",(R),AL-02,(D),False,True,Flip,R+2.2,62.150,37.8,62.2,37.8%,62.2%,12.304433


In [122]:
data['disp_24_to_26_swing'] = data['swing_24_to_26'].map(lambda x: f'D+{x:.1f}' if x > 0 else f'R+{abs(x):.1f}')
data.head(2)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,state,district_x,cvap_white_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_black_pct,college,lean_20,lean_24,pvi,urban_pct,dem_2p_24,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,fips,geoid,demo_cluster,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,district_y,curr_party,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26,disp_24_to_26_swing
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",2364393.34,3310044.97,5674438.31,41.667443,58.332557,Alaska,AK-00,64.463948,6.051383,6.481397,12.640423,3.077471,30.750000,-7.531684,-6.096509,-6.455302,64.899487,43.153437,0.987401,-8.650667,0,1,0.0,0.0,0.0,0.0,Pacific,2026,0.0,0.0,0.0,2,200,5,1736.175794,0.0,0.0,0.0,-4.259937,-8.332557,-1,-16.665114,46.396984,3.601511,15.775,1,39.326982,53.464556,0.065,Likely R,"<b style=""color:#792ba6;"">Bill Hill (I)</b> vs...",(R),AK-AL,(R),True,False,,R+7.2,84.225,15.8,84.2,15.8%,84.2%,6.487092,D+6.5
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",184951.75,739240.65,924192.40,20.012256,79.987744,Alabama,AL-01,69.156017,2.677114,1.299003,0.469439,24.385648,17.996871,-17.482017,-17.422182,-17.437141,68.324941,31.827764,0.987401,-8.650667,0,0,0.0,0.0,0.0,0.0,East South Central,2026,0.0,0.0,0.0,1,101,5,400.490394,0.0,0.0,0.0,-26.223615,-29.987744,0,-59.975488,37.326339,3.632273,0.020,2,30.246696,44.405075,0.000,Safe R,"<b style=""color:#366bbf;"">Clyde Jones Jr. (D)<...",(R),AL-01,(R),True,False,,R+25.3,99.980,0.0,100.0,<1%,>99%,10.997150,D+11.0


In [123]:
data['geoid']

0       200
1       101
2       102
3       103
4       104
       ... 
413    5507
414    5508
415    5401
416    5402
417    5600
Name: geoid, Length: 418, dtype: int64

In [124]:
dem_uclen = dem_uncont.shape[0]
dem_uncont['rating'] = np.full(dem_uclen, 'Safe D')
dem_uncont['chance'] = np.full(dem_uclen, 100)
dem_uncont['y_pred'] = np.full(dem_uclen, 100)
dem_uncont['swing_24_to_26'] = np.full(dem_uclen, float('nan')) # Placeholder
dem_uncont['disp_24_to_26_swing'] = np.full(dem_uclen, float('nan')) # Placeholder
dem_uncont['disp_dem_chance'] = np.full(dem_uclen, '100%')
dem_uncont['disp_rep_chance'] = np.full(dem_uclen, '0%')
dem_uncont['projected_2p_margin'] = np.full(dem_uclen, 'D+100')
dem_uncont['flip_indic'] = np.full(dem_uclen, '')
dem_uncont = dem_uncont.drop(['Unnamed: 0'], axis=1)
dem_uncont['matchup'] = dem_uncont[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                                      x['dem_inc_any'], x['rep_inc_any']), axis=1)
dem_uncont.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fips,geoid,rating,chance,y_pred,swing_24_to_26,disp_24_to_26_swing,disp_dem_chance,disp_rep_chance,projected_2p_margin,flip_indic,matchup
0,CA-04,Mike Thompson/Eric Jones,Not Contested,True,False,CA,4,6,604,Safe D,100,100,NaN,NaN,100%,0%,D+100,,"<b style=""color:#366bbf;"">Mike Thompson/Eric J..."
1,CA-07,Doris Matsui/Mai Vang,Not Contested,True,False,CA,7,6,607,Safe D,100,100,NaN,NaN,100%,0%,D+100,,"<b style=""color:#366bbf;"">Doris Matsui/Mai Van..."
2,CA-11,Scott Weiner/Connie Chan,Not Contested,False,False,CA,11,6,611,Safe D,100,100,NaN,NaN,100%,0%,D+100,,"<b style=""color:#366bbf;"">Scott Weiner/Connie ..."
3,CA-12,Lateefah Simon,Not Contested,True,False,CA,12,6,612,Safe D,100,100,NaN,NaN,100%,0%,D+100,,"<b style=""color:#366bbf;"">Lateefah Simon* (D)<..."
4,CA-14,Aisha Wahab/Melissa Hernandez,Not Contested,True,False,CA,14,6,614,Safe D,100,100,NaN,NaN,100%,0%,D+100,,"<b style=""color:#366bbf;"">Aisha Wahab/Melissa ..."


In [125]:
rep_uclen = rep_uncont.shape[0]
rep_uncont['rating'] = np.full(rep_uclen, 'Safe R')
rep_uncont['chance'] = np.full(rep_uclen, 0)
rep_uncont['y_pred'] = np.full(rep_uclen, 0)
rep_uncont['swing_24_to_26'] = np.full(rep_uclen, float('nan')) # Placeholder
rep_uncont['disp_24_to_26_swing'] = np.full(rep_uclen, float('nan')) # Placeholder
rep_uncont['disp_dem_chance'] = np.full(rep_uclen, '0%')
rep_uncont['disp_rep_chance'] = np.full(rep_uclen, '100%')
rep_uncont['projected_2p_margin'] = np.full(rep_uclen, 'R+100')
rep_uncont['flip_indic'] = np.full(rep_uclen, '')
rep_uncont = rep_uncont.drop(['Unnamed: 0'], axis=1)
rep_uncont['matchup'] = rep_uncont[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                                       x['dem_inc_any'], x['rep_inc_any']), axis=1)
rep_uncont.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fips,geoid,rating,chance,y_pred,swing_24_to_26,disp_24_to_26_swing,disp_dem_chance,disp_rep_chance,projected_2p_margin,flip_indic,matchup
0,CA-40,Not Contested,Young Kim/Ken Calvert,False,True,CA,40,6,640,Safe R,0,0,NaN,NaN,0%,100%,R+100,,"<b style=""color:#366bbf;"">Not Contested (D)</b..."


In [126]:
incl_cols = ['cd', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'y_pred', 'flip_indic', 'geoid', 'matchup', 'swing_24_to_26', 'disp_24_to_26_swing']
disp_data = pd.concat([data[incl_cols], dem_uncont[incl_cols], rep_uncont[incl_cols]], axis=0)
disp_data.shape

(435, 16)

In [127]:
tab_data = data[['cd', 'dem_cand', 'rep_cand', 'rating', 'disp_dem_chance', 'disp_rep_chance', 'projected_2p_margin', 'disp_24_to_26_swing',
                 'tipping_point_prob']]
for party in ['dem', 'rep']:
    tab_data[f'disp_{party}_chance'] = tab_data[f'disp_{party}_chance'].map(lambda x: f'<p style="color:{'blue' if party == 'dem' else 'red'};">{x}</p>')
    tab_data[f'{party}_cand'] = tab_data[f'{party}_cand'].map(lambda x: f'{x} (Ind)' if x in ['Bill Hill', 'Kevin Kiley'] else x)
tab_data = tab_data.rename({
    'cd': 'District',
    'dem_cand': 'Democrat',
    'rep_cand': 'Republican',
    'rating': 'Rating',
    'disp_dem_chance': 'Dem Chance',
    'disp_rep_chance': 'Rep Chance',
    'projected_2p_margin': 'Projected Margin',
    'tipping_point_prob': 'Tipping Point Chance',
    'disp_24_to_26_swing': 'Swing from 2024 Pres'
}, axis=1)
tab_data.head()

,District,Democrat,Republican,Rating,Dem Chance,Rep Chance,Projected Margin,Swing from 2024 Pres,Tipping Point Chance
0,AK-AL,Bill Hill (Ind),Nick Begich,Likely R,"<p style=""color:blue;"">15.8%</p>","<p style=""color:red;"">84.2%</p>",R+7.2,D+6.5,0.065
1,AL-01,Clyde Jones Jr.,Jerry Carl,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+25.3,D+11.0,0.000
2,AL-02,Shomari Figures,Rhett Marques,Tilt R,"<p style=""color:blue;"">37.8%</p>","<p style=""color:red;"">62.2%</p>",R+2.2,D+12.3,0.840
3,AL-03,Lee McInnis,Mike Rogers,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+39.3,D+7.7,0.000
4,AL-04,Amanda Pusczek,Robert Aderholt,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+58.9,D+8.1,0.000


In [128]:
data['rep_pred'] = data['y_pred'].map(lambda x: 100 - x)
data['projected_2p_margin_number'] = data['rep_pred'] - data['y_pred']
data['proj_seat_lean'] = data['projected_2p_margin_number'] - data['generic_ballot_avg']
sv_bias = float(data.sort_values('tipping_point_prob', ascending=False).reset_index().loc[0, 'proj_seat_lean']) # Seats-votes bias, + = R, - = D
sv_bias

4.745236077978058

In [129]:
topline_stats = pd.read_csv('display_data/topline_stats.csv')
if 'sv_bias' in topline_stats['vars'].values:
    topline_stats = topline_stats[topline_stats['vars'] != 'sv_bias']
topline_stats = pd.concat([topline_stats, pd.DataFrame({'vars': ['sv_bias'], 'x': sv_bias})], axis=0)
topline_stats.to_csv('display_data/topline_stats.csv')
topline_stats

,vars,x
0,means_seats_tot,241.977950
1,chamber_win_chance,83.010000
0,sv_bias,4.745236


In [130]:
mean_seats_tot = topline_stats[topline_stats['vars'] == 'means_seats_tot']['x'].values[0]
chamber_win_chance = topline_stats[topline_stats['vars'] == 'chamber_win_chance']['x'].values[0]

In [131]:
chances = topline_stats[topline_stats['vars'] == 'chamber_win_chance'].set_index(['vars']).T
chances['Republicans'] = chances['chamber_win_chance'].map(lambda x: 100 - x)
chances = chances.rename({'chamber_win_chance': 'Democrats'}, axis=1).T.reset_index()
chances = chances.rename({'x': 'Win Probability'}, axis=1)
seats = topline_stats[topline_stats['vars'] == 'means_seats_tot'].set_index(['vars']).T
seats['Republicans'] = seats['means_seats_tot'].map(lambda x: 435 - x)
seats = seats.rename({'means_seats_tot': 'Democrats'}, axis=1).T.reset_index().rename({'x': 'Seat Share'}, axis=1)
summary_stats = pd.merge(left=chances, right=seats, on='vars', how='inner').rename({'vars': 'Party'}, axis=1)
summary_stats.to_csv('display_data/summary_stats.csv')
summary_stats

,Party,Win Probability,Seat Share
0,Democrats,83.01,241.97795
1,Republicans,16.99,193.02205


In [132]:
## Alert when there's a rating change
rating_ts = output_ts[output_ts['type'] == 'chance']
rating_ts['rating'] = rating_ts['y'].map(get_rating)
rating_ts['date'] = pd.to_datetime(rating_ts['date'])
rating_ts.head()

today_date = rating_ts['date'].max()
yesterday = rating_ts[rating_ts['date'] != today_date]['date'].max()

today_ratings = rating_ts[rating_ts['date'] == today_date]
yester_ratings = rating_ts[rating_ts['date'] == yesterday]
compare_df = pd.merge(left=today_ratings, right=yester_ratings, on='geo', how='left', suffixes=['_t', '_y'])
compare_df.head()

,date_t,y_t,geo,type_t,rating_t,date_y,y_y,type_y,rating_y
0,2026-11-02,83.010,US House,chance,Likely D,2026-11-01,82.425,chance,Likely D
1,2026-11-02,15.775,AK-AL,chance,Likely R,2026-11-01,15.185,chance,Likely R
2,2026-11-02,0.020,AL-01,chance,Safe R,2026-11-01,0.020,chance,Safe R
3,2026-11-02,37.850,AL-02,chance,Tilt R,2026-11-01,37.005,chance,Tilt R
4,2026-11-02,0.000,AL-03,chance,Safe R,2026-11-01,0.000,chance,Safe R


In [133]:
## This is the cell that actually alerts when there's a rating change
if any(compare_df['rating_t'] != compare_df['rating_y']):
    changes_df = compare_df[compare_df['rating_t'] != compare_df['rating_y']][['geo', 'rating_y', 'rating_t']]
    print(changes_df)
else:
    print("No rating changes today!")

       geo rating_y       rating_t
17   AZ-05   Safe R  Very Likely R
122  IA-01   Tossup         Tilt D
258  NY-01   Lean R         Tilt R


In [134]:
for i in range(changes_df.shape[0]):
    seat = changes_df.iloc[i, :]['geo']
    rating_y = changes_df.iloc[i, :]['rating_y']
    rating_t = changes_df.iloc[i, :]['rating_t']
    print(f'{seat}: {rating_y} --> {rating_t}')

AZ-05: Safe R --> Very Likely R
IA-01: Tossup --> Tilt D
NY-01: Lean R --> Tilt R


In [135]:
changes_df.shape

(3, 3)

In [136]:
if any(compare_df['rating_t'] != compare_df['rating_y']):
    for i in range(changes_df.shape[0]):
        seat = changes_df.iloc[i, :]['geo']
        rating_y = changes_df.iloc[i, :]['rating_y']
        rating_t = changes_df.iloc[i, :]['rating_t']
        print(f'{seat}: {rating_y} --> {rating_t}')

AZ-05: Safe R --> Very Likely R
IA-01: Tossup --> Tilt D
NY-01: Lean R --> Tilt R


In [137]:
## Mini-widget: what is the chance of Dems/Reps winning House if X congressional district is won by Dems/Reps?
def det_winner(post, cd, dem_winner=True):
    '''
    :param post: Posterior simulation matrix.
    :param cd: Congressional district.
    :param dem_winner: Whether or not we set Dems or Reps to win `cd`. True if Dem win is fixed, False if Rep win is fixed.
    '''
    winners = []
    seats = []
    post_ut = post.T
    if dem_winner:
        post_ut = post_ut[post_ut[cd] > 0]
    else:
        post_ut = post_ut[post_ut[cd] < 0]
    post_case = post_ut.T
    for i in range(post_case.shape[0]):
        sim = post_case.iloc[:, i]
        dem_seats = np.sum(sim > 0)
        if dem_seats >= 218 - dem_uncont.shape[0]: # Democrats win House in this draw
            winners.append(1)
        else: # Republicans win House in this draw
            winners.append(0)
        seats.append(dem_seats)
    dem_win_chance = np.mean(np.array(winners)) * 100
    dem_mean_seats = np.mean(np.array(seats)) + dem_uncont.shape[0]
    return dem_win_chance, dem_mean_seats

cd = 'MI-10'
dem_winner = True
det_winner(post, cd, dem_winner)

(np.float64(97.60765550239235), np.float64(253.02631578947367))

In [138]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
tab_data.to_csv('display_data/table_display_data.csv')
data.to_csv('display_data/all_data.csv')